# Notebook 4: Evaluation & Comparative Analysis
**Authors:** David Ho, Mahmoud Abdulkareem

> **Prerequisite:** Run Notebooks 1, 2, and 3 first.

## Step 1: Mount Drive + Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — go to Runtime > Change runtime type > T4 GPU')

## Step 2: Install Dependencies

In [ ]:
!pip install -q ultralytics transformers pycocotools pandas

In [ ]:
import json, time, random, zipfile, shutil
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
from transformers import DetrForObjectDetection, DetrImageProcessor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Step 3: Config

In [ ]:
DRIVE_ROOT    = Path('/content/drive/MyDrive/vehicle_detection')
DRIVE_BACKUP  = DRIVE_ROOT / 'processed'
YOLO_RESULTS  = DRIVE_ROOT / 'results' / 'yolo'
DETR_RESULTS  = DRIVE_ROOT / 'results' / 'detr'
OUT_DIR       = DRIVE_ROOT / 'results' / 'comparison'
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_BDD  = Path('/content/bdd100k')
LOCAL_DATA = Path('/content/data')

VEHICLE_CLASSES = ['car', 'truck', 'bus', 'motorcycle']
COLORS_MODEL    = ['#1565C0', '#E53935']
COLORS_CLS      = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']

assert (YOLO_RESULTS / 'metrics.json').exists(), 'Run Notebook 2 first.'
assert (DETR_RESULTS / 'metrics.json').exists(), 'Run Notebook 3 first.'
print('Config OK')

## Step 4: Session Setup
Unzips images locally and restores label files. Run every new Colab session.

In [ ]:
DRIVE_ZIP = DRIVE_ROOT / 'downloads' / 'solesensei_bdd100k.zip'

if not LOCAL_BDD.exists():
    print('Unzipping BDD100K to local disk (~5 min)...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall('/content/')
    print('Unzip done.')

if not (LOCAL_BDD / 'images').exists() and (LOCAL_BDD / 'bdd100k' / 'images').exists():
    LOCAL_BDD = LOCAL_BDD / 'bdd100k'

if not LOCAL_DATA.exists():
    print('Restoring label files from Drive...')
    shutil.copytree(DRIVE_BACKUP, LOCAL_DATA)
    print('Done.')

COCO_ANN = LOCAL_DATA / 'coco' / 'annotations'
print('Session setup complete.')

## Step 5: Load Metrics

In [ ]:
yolo_m = json.loads((YOLO_RESULTS / 'metrics.json').read_text())
detr_m = json.loads((DETR_RESULTS / 'metrics.json').read_text())
print('YOLO:', json.dumps(yolo_m, indent=2))
print('DETR:', json.dumps(detr_m, indent=2))

## Step 6: Comparison Table

In [ ]:
rows = [
    ('mAP@50',             yolo_m['map50'],                            detr_m['map50']),
    ('mAP@50-95',          yolo_m['map50_95'],                         detr_m['map50_95']),
    ('Precision',          yolo_m['precision'],                        detr_m['precision']),
    ('Recall',             yolo_m['recall'],                           detr_m['recall']),
    ('F1-Score',           yolo_m['f1'],                               detr_m['f1']),
    ('Inference (ms/img)', yolo_m['inference_ms'],                     detr_m['inference_ms']),
    ('FPS',                yolo_m['fps'],                              detr_m['fps']),
    ('Params (M)',         round(yolo_m['trainable_params']/1e6, 2),   round(detr_m['trainable_params']/1e6, 2)),
    ('Time/Epoch (s)',     yolo_m['time_per_epoch_s'],                 detr_m['time_per_epoch_s']),
]
df = pd.DataFrame(rows, columns=['Metric', 'YOLOv8s', 'DETR-ResNet50']).set_index('Metric')
print(df.to_string())
df.to_csv(OUT_DIR / 'comparison_table.csv')
print('\nSaved comparison_table.csv')

## Step 7: Accuracy Bar Charts

In [ ]:
models      = ['YOLOv8s', 'DETR-ResNet50']
acc_metrics = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'F1-Score']
fig, axes   = plt.subplots(1, len(acc_metrics), figsize=(18, 5))
fig.suptitle('Detection Accuracy Comparison', fontsize=14)

for ax, metric in zip(axes, acc_metrics):
    vals = [df.loc[metric, m] for m in models]
    bars = ax.bar(models, vals, color=COLORS_MODEL, width=0.5, edgecolor='white')
    ax.set_title(metric, fontsize=11); ax.set_ylim(0, 1); ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'accuracy_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

## Step 8: Efficiency Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Computational Efficiency Comparison', fontsize=14)
eff = [('Inference (ms/img)', False), ('FPS', True), ('Time/Epoch (s)', False)]

for ax, (metric, higher_better) in zip(axes, eff):
    vals   = [df.loc[metric, m] for m in models]
    winner = np.argmax(vals) if higher_better else np.argmin(vals)
    colors = [COLORS_MODEL[i] if i == winner else '#90A4AE' for i in range(2)]
    bars   = ax.bar(models, vals, color=colors, width=0.5, edgecolor='white')
    ax.set_title(metric, fontsize=11); ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'efficiency_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

## Step 9: Precision-Recall Curves

In [ ]:
yolo_best = YOLO(str(YOLO_RESULTS / 'train' / 'weights' / 'best.pt'))
detr_best = DetrForObjectDetection.from_pretrained(DETR_RESULTS / 'best_model').to(DEVICE).eval()
detr_proc = DetrImageProcessor.from_pretrained(DETR_RESULTS / 'best_model')

with open(COCO_ANN / 'instances_test.json') as f:
    test_ann = json.load(f)
id2label      = {cat['id']: cat['name'] for cat in test_ann['categories']}
img_name2id   = {img['file_name']: img['id'] for img in test_ann['images']}
coco_gt_test  = COCO(str(COCO_ANN / 'instances_test.json'))

# file_name is stored as relative path from LOCAL_BDD
test_img_paths = [LOCAL_BDD / img['file_name'] for img in test_ann['images']]
print(f'Test images: {len(test_img_paths):,}')

In [ ]:
def collect_yolo_preds(model, img_paths, name2id, conf=0.01):
    preds = []
    for p in img_paths:
        rel_name = str(p.relative_to(LOCAL_BDD))
        img_id = name2id.get(rel_name)
        if img_id is None: continue
        res = model.predict(str(p), imgsz=640, conf=conf, device=DEVICE, verbose=False)[0]
        for box in res.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            preds.append({'image_id': img_id, 'category_id': int(box.cls[0]),
                          'bbox': [x1, y1, x2-x1, y2-y1], 'score': float(box.conf[0])})
    return preds


def collect_detr_preds(model, proc, img_paths, name2id, conf=0.01):
    preds = []
    with torch.no_grad():
        for p in img_paths:
            rel_name = str(p.relative_to(LOCAL_BDD))
            img_id = name2id.get(rel_name)
            if img_id is None: continue
            img = Image.open(p).convert('RGB')
            enc = proc(images=img, return_tensors='pt').to(DEVICE)
            res = proc.post_process_object_detection(
                model(**enc), threshold=conf, target_sizes=[img.size[::-1]])[0]
            for score, label, box in zip(res['scores'], res['labels'], res['boxes']):
                x1, y1, x2, y2 = box.tolist()
                preds.append({'image_id': img_id, 'category_id': label.item(),
                              'bbox': [x1, y1, x2-x1, y2-y1], 'score': round(score.item(), 4)})
    return preds


def pr_curve(coco_gt, preds, iou_thresh=0.5):
    gt_by_img = defaultdict(list)
    for ann in coco_gt.dataset['annotations']: gt_by_img[ann['image_id']].append(ann)
    total_gt = sum(len(v) for v in gt_by_img.values())
    precs, recs = [], []
    for thresh in sorted({p['score'] for p in preds}, reverse=True):
        filtered    = [p for p in preds if p['score'] >= thresh]
        pred_by_img = defaultdict(list)
        for p in filtered: pred_by_img[p['image_id']].append(p)
        TP = FP = 0
        for img_id, gts in gt_by_img.items():
            matched = [False] * len(gts)
            for pred in sorted(pred_by_img[img_id], key=lambda x: -x['score']):
                best_iou, best_j = 0, -1
                for j, gt in enumerate(gts):
                    if matched[j] or gt['category_id'] != pred['category_id']: continue
                    b1, b2 = pred['bbox'], gt['bbox']
                    inter = max(0, min(b1[0]+b1[2], b2[0]+b2[2]) - max(b1[0], b2[0])) * \
                            max(0, min(b1[1]+b1[3], b2[1]+b2[3]) - max(b1[1], b2[1]))
                    union = b1[2]*b1[3] + b2[2]*b2[3] - inter
                    iou = inter / union if union > 0 else 0
                    if iou > best_iou: best_iou, best_j = iou, j
                if best_iou >= iou_thresh: TP += 1; matched[best_j] = True
                else: FP += 1
        FN = total_gt - TP
        precs.append(TP / (TP + FP + 1e-9))
        recs.append(TP  / (TP + FN + 1e-9))
    return np.array(recs), np.array(precs)


print('Collecting YOLO predictions...')
yolo_preds = collect_yolo_preds(yolo_best, test_img_paths, img_name2id)
print(f'  {len(yolo_preds):,}')
print('Collecting DETR predictions...')
detr_preds = collect_detr_preds(detr_best, detr_proc, test_img_paths, img_name2id)
print(f'  {len(detr_preds):,}')

In [ ]:
yolo_rec, yolo_prec = pr_curve(coco_gt_test, yolo_preds)
detr_rec, detr_prec = pr_curve(coco_gt_test, detr_preds)
yolo_ap = np.trapz(yolo_prec[np.argsort(yolo_rec)], np.sort(yolo_rec))
detr_ap = np.trapz(detr_prec[np.argsort(detr_rec)], np.sort(detr_rec))

plt.figure(figsize=(8, 6))
plt.plot(yolo_rec, yolo_prec, color=COLORS_MODEL[0], lw=2, label=f'YOLOv8s (AP={yolo_ap:.3f})')
plt.plot(detr_rec, detr_prec, color=COLORS_MODEL[1], lw=2, ls='--', label=f'DETR-ResNet50 (AP={detr_ap:.3f})')
plt.xlabel('Recall', fontsize=12); plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve (all classes, IoU=0.5)', fontsize=13)
plt.legend(fontsize=11); plt.grid(alpha=0.3); plt.xlim([0,1]); plt.ylim([0,1])
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'pr_curve.png'), dpi=120)
plt.show()

## Step 10: Side-by-Side Detections

In [ ]:
CONF_VIZ   = 0.4
sample     = random.sample(test_img_paths, min(4, len(test_img_paths)))
gt_by_name = defaultdict(list)
for ann in test_ann['annotations']:
    fname = next(i['file_name'] for i in test_ann['images'] if i['id'] == ann['image_id'])
    gt_by_name[fname].append(ann)

fig, axes = plt.subplots(len(sample), 3, figsize=(18, 5*len(sample)))
fig.suptitle('Ground Truth  |  YOLOv8s  |  DETR-ResNet50', fontsize=13)

for row, img_path in enumerate(sample):
    img      = Image.open(img_path).convert('RGB')
    rel_name = str(img_path.relative_to(LOCAL_BDD))

    # GT
    ax = axes[row][0]; ax.imshow(img)
    for ann in gt_by_name[rel_name]:
        x, y, w, h = ann['bbox']
        cls = id2label.get(ann['category_id'], '?')
        col = COLORS_CLS[VEHICLE_CLASSES.index(cls)] if cls in VEHICLE_CLASSES else 'white'
        ax.add_patch(patches.Rectangle((x,y), w, h, lw=2, ec=col, fc='none'))
        ax.text(x, y-4, cls, color=col, fontsize=7, fontweight='bold')
    ax.set_title('Ground Truth', fontsize=10); ax.axis('off')

    # YOLO
    ax = axes[row][1]; ax.imshow(img)
    res = yolo_best.predict(str(img_path), imgsz=640, conf=CONF_VIZ, device=DEVICE, verbose=False)[0]
    for box in res.boxes:
        x1,y1,x2,y2 = box.xyxy[0].tolist(); cls_id = int(box.cls[0])
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec=COLORS_CLS[cls_id],fc='none'))
        ax.text(x1,y1-4,f"{VEHICLE_CLASSES[cls_id]} {float(box.conf[0]):.2f}",
                color=COLORS_CLS[cls_id],fontsize=7,fontweight='bold')
    ax.set_title('YOLOv8s', fontsize=10); ax.axis('off')

    # DETR
    ax = axes[row][2]; ax.imshow(img)
    enc = detr_proc(images=img, return_tensors='pt').to(DEVICE)
    with torch.no_grad(): out = detr_best(**enc)
    res = detr_proc.post_process_object_detection(out, threshold=CONF_VIZ, target_sizes=[img.size[::-1]])[0]
    for score, label, box in zip(res['scores'], res['labels'], res['boxes']):
        x1,y1,x2,y2 = box.tolist()
        cls  = id2label.get(label.item(), '?')
        idx  = VEHICLE_CLASSES.index(cls) if cls in VEHICLE_CLASSES else 0
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec=COLORS_CLS[idx],fc='none'))
        ax.text(x1,y1-4,f'{cls} {score:.2f}',color=COLORS_CLS[idx],fontsize=7,fontweight='bold')
    ax.set_title('DETR-ResNet50', fontsize=10); ax.axis('off')

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'side_by_side.png'), dpi=100, bbox_inches='tight')
plt.show()

## Step 11: Final Summary

In [ ]:
print('=' * 55)
print('FINAL COMPARISON SUMMARY')
print('=' * 55)
print(df.to_string())
print('\nAll outputs saved to:', OUT_DIR)

## Discussion

| Dimension | Expected Winner | Notes |
|---|---|---|
| Accuracy (mAP@50) | TBD | DETR may excel in crowded/occluded scenes |
| Inference speed | **YOLOv8s** | Single-pass CNN vs. Transformer attention |
| Model size | **YOLOv8s** | DETR backbone adds ~25M extra params |
| Training speed | **YOLOv8s** | DETR needs more epochs to converge |

**Recommendations:**
- Real-time deployment → **YOLOv8**: faster inference, easy ONNX/TensorRT export
- Accuracy-critical offline tasks → **DETR**: global attention handles complex scenes better